In [8]:
from pathlib import Path
import os
from Bio import SeqIO
import pandas as pd

# 自动定位仓库根
current = Path(os.path.abspath("."))
while not (current / "week1_environment_check").exists():
    current = current.parent
fasta_path = str(current / "week1_environment_check" / "data" / "demo_sequences.fasta")

# 输出路径
yourname = "dujiayi"
output_dir = f"submissions/{yourname}/week1/results"
output_file = "week1_sequence_stats.csv"
output_path = os.path.join(output_dir, output_file)

# 确保输出目录存在
os.makedirs(output_dir, exist_ok=True)

# 读取并统计序列
records = list(SeqIO.parse(fasta_path, "fasta"))
result_list = []

for i in records:
    seq_id = i.id
    seq = str(i.seq)
    length = len(seq)
    gc = seq.count('G') + seq.count('C')
    gc_pct = (gc / length) * 100 if length > 0 else 0.0

    result_list.append({
        "sequence_id": seq_id,
        "length": length,
        "GC_count": gc,
        "GC_percent": round(gc_pct, 2)
    })

# 保存结果
df = pd.DataFrame(result_list)
df.to_csv(output_path, index=False)
# print(f"结果已保存到：{output_path}")

In [1]:
import os
import pandas as pd

input_file = "../../../../week1_environment_check/data/demo_measurements.csv"
name = "dujiayi"
output_folder = "../results"  
output_file = os.path.join(output_folder, "week1_merged_table.csv")

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

data = pd.read_csv(input_file)
# 计算标准化荧光
new_column = []
for i in range(len(data)):
    od = data["od600"][i]
    fluo = data["fluorescence"][i]
    if od == 0:
        new_column.append(0)
    else:
        new_column.append(fluo / od)

data["normalized_fluorescence"] = new_column

data.to_csv(output_file, index=False)


In [2]:
import os
import pandas as pd

# 输入1：FASTA序列统计表（任务1输出）
seq_stats_path = "../../../../submissions/dujiayi/week1/results/week1_sequence_stats.csv"
# 输入2：CSV测量数据表（含标准化荧光，任务2输出）
measure_path = "../../../../submissions/dujiayi/week1/results/week1_merged_table.csv"
# 输出：最终合并表（任务3要求）
your_name = "dujiayi"
output_dir = f"../../../../submissions/{your_name}/week1/results"
output_file = os.path.join(output_dir, "week1_merged_table.csv")

if not os.path.exists(output_dir):
    os.makedirs(output_dir)


df_seq = pd.read_csv(seq_stats_path)
df_measure = pd.read_csv(measure_path)

df_merged = pd.merge(
    left=df_seq,        # 序列表（少行）
    right=df_measure,  # 测量表（多行）
    on="sequence_id",  # 合并依据
    how="right"        # 多对一：以测量表为准
)

df_merged.to_csv(output_file, index=False)

In [3]:
import os
import pandas as pd
import matplotlib.pyplot as plt

merged_path = "../../../../submissions/dujiayi/week1/results/week1_merged_table.csv"
your_name = "dujiayi"
fig_dir = f"../../../../submissions/{your_name}/week1/figures"

if not os.path.exists(fig_dir):
    os.makedirs(fig_dir)

df = pd.read_csv(merged_path)

plt.figure(figsize=(10, 5))
plt.bar(
    x=df["sample_id"],
    height=df["normalized_fluorescence"],
    color="#2e8b57"
)
plt.title("Normalized Fluorescence per Sample")
plt.xlabel("Sample ID")
plt.ylabel("Normalized Fluorescence")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "week1_normalized_fluorescence_bar.png"), dpi=300)
plt.close()

# 图2：control 与 treatment 分组比较图
plt.figure(figsize=(8, 5))
group_stats = df.groupby("group")["normalized_fluorescence"].agg(["mean", "std"]).reset_index()

plt.bar(
    x=group_stats["group"],
    height=group_stats["mean"],
    yerr=group_stats["std"],
    capsize=5,
    color=["#87ceeb", "#ff6347"]
)
plt.title("Normalized Fluorescence: Control vs Treatment")
plt.xlabel("Group")
plt.ylabel("Mean Normalized Fluorescence")
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "week1_group_comparison.png"), dpi=300)
plt.close()

# 图3：GC_percent 与 normalized_fluorescence 散点图
plt.figure(figsize=(8, 5))
plt.scatter(
    x=df["GC_percent"],  
    y=df["normalized_fluorescence"],
    c="#9370db",
    s=80,
    alpha=0.7
)
plt.title("GC Content vs Normalized Fluorescence")
plt.xlabel("GC Percent (%)")
plt.ylabel("Normalized Fluorescence")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "week1_gc_vs_expression.png"), dpi=300)
plt.close()
